# Processing workflow for collecting data on V-BReE performance using the MMLU-PRO dataset.

## 1. Adding the V-BReE framework to a workspace

### Import the V-BReE module from Github
To install the V-BReE module, clone the git repository into the colab file system.

---

*If you run into problems with folder naming, you should be able to delete or rename the existing folder and then create a fresh clone*

In [15]:
from pathlib import Path

vbree_path = Path("v_bree.py")

if not vbree_path.is_file():
  !wget https://raw.githubusercontent.com/JasonIves/V-BReE/main/v_bree.py

### Import the module

You can now import the *v_bree* module.

In [16]:
import v_bree

## 2. Configuring the V-BReE framework workspace

### Store inference connection API key
Modeling and inference connections often require an API key.  That key can be stored in the "Secrets" section of the sidebar, given a name, and then referenced using the *userdata* library.  This example represents a HuggingFace Inference Client key, but other services should work similarly.

Do not store API keys directly in code.

Once the token is stored you will need to give the notebook permission to access it.

In [17]:
from google.colab import userdata

hfToken = userdata.get('HF_TOKEN')

### Define inference connection client

Define a client connection that uses the OpenAI API format.  Primary testing for this module was done using the [Hugging Face Inference Client](https://huggingface.co/docs/huggingface_hub/en/package_reference/inference_client), but other clients utilizing the OpenAI API format should work as well.


In [18]:
from huggingface_hub import InferenceClient

client = InferenceClient(
              provider = "auto",
              api_key = hfToken
          )

## 3. Working with the V-BReE framework

### General system configuration

As usual, finishing outfitting the workspace with the necessary Python tools.


In [19]:
import json
import re

import pandas as pd
from datasets import load_dataset

### Load data mirroring ICE test data set

For this processing run, we will load a data set that mirrors the data set used for the ICE ensemble testing.

- 3000 cases
- 250 each from 12 separate categories
  - biology
  - business
  - chemistry
  - computers
  - economics
  - history
  - law
  - math
  - philosophy
  - physics
  - engineering
  - psychology

In [20]:
##STARTING AND ENDING CASES FOR BATCH PROCESSING
sample_start = 500
sample_end = 1000

categories = ["biology", "business", "chemistry", "computer science", "economics", "history", "law", "math", "philosophy", "physics", "engineering", "psychology"]

dataset = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
data = dataset.to_pandas()

sample = pd.DataFrame()

for category in categories:
  cat_data = data[data["category"] == category]
  cat_sample = cat_data.sample(n = 250, random_state = 99)

  sample = pd.concat([sample, cat_sample])

sample = sample.reset_index(drop = True)

# sample.to_csv("mmlu_pro_12_cat_250_sample.csv", index = False)  ##ALREADY CAPTURED FULL SAMPLE

sample = sample.iloc[sample_start:sample_end]

display(sample.head(10))



,question_id,question,options,answer,answer_index,cot_content,category,src
500,4205,For the following second-order reversible reac...,"[(k_b / k_a) = K, K = k_a + k_b, K = (k_b * k_...",D,3,,chemistry,stemez-PhysicalChemistry
501,4091,The following data show how the standard molar...,"[49.2 $\mathrm{~kJ} \mathrm{~mol}^{-1}$, 85.7 ...",E,4,,chemistry,scibench-matter
502,3747,The density of KF is 2.48 g/cm^3. The solid is...,"[4.2 × 10^23 formula units/cube, 3.5 × 10^23 f...",I,8,,chemistry,stemez-Chemistry
503,4323,A snowball at - 15°C is thrown at a large tree...,"[925 m/s, 1100 m/s, 720 m/s, 950 m/s, 892 m/s,...",F,5,,chemistry,stemez-Chemistry
504,3953,0.001 mol ofNaOHis added to 100 ml. of a solut...,"[5.20, 4.68, 5.10, 4.74, 4.76, 5.00, 4.70, 4.9...",E,4,,chemistry,stemez-Chemistry
505,4537,Determine the de Broglie wavelength of an elec...,"[1.326 Å, 0.426 Å, 1.626 Å, 1.026 Å, 0.726 Å, ...",G,6,,chemistry,stemez-Chemistry
506,4604,What is the maximum number of phases that can ...,"[6, 9, 2, 4, 10, 8, 7, 3, 1, 5]",J,9,,chemistry,ori_mmlu-college_chemistry
507,4347,What is the limiting high-temperature molar he...,"[4.5R, 2.2R, 1R, 4R, 2R, 2.5R, 5R, 3R, 3.5R, 1...",I,8,,chemistry,ori_mmlu-college_chemistry
508,3927,A 0.100 l container maintained at constant tem...,"[5.0 x 10^9 molecules, 1 x 10^-6 l, 2.5 x 10^9...",D,3,,chemistry,stemez-Chemistry
509,4109,Find the degree of ionization of 0.05 M NH_3 i...,"[2.00 × 10^-4, 6.20 × 10^-3, 9.00 × 10^-4, 3.5...",H,7,,chemistry,stemez-Chemistry


### Configure the data for V-BReE compatibility

A V-BReE compatibile dataset has 4 key components.  Each must be provided, although the domain can be set to a dummy value without impacting ensemble processing.

- An identifier that is unique for each row. Ex: "*12893*"
- The question being asked of the ensemble.  This should be a string. Ex: "*What is 2 + 2?*"
- The possible choices, formatted as a list. Ex: "*[3, 4, 5]*"
  - Actual choices are not required. But an empty list, Ex: "*[]*", should still be submitted in the designated choices column.  When detected the ensemble will proceed in free-response mode instead of MCQ mode.
- Question domain.  Not required. For conveyence to results data. Ex: "*math*"

Any other data submitted to the V-BReE ensemble will be ignored.

In [21]:
##REFORMAT OPTIONS COLUMN TO ENSURE LIST STRUCTURE
choice_pattern = re.compile(r"\'(.*?)\'")

sample["options"] = sample["options"].astype(str)
sample["options"] = sample["options"].apply(lambda x: re.findall(choice_pattern, x) if pd.notnull(x) else [])

display(sample)

,question_id,question,options,answer,answer_index,cot_content,category,src
500,4205,For the following second-order reversible reac...,"[(k_b / k_a) = K, K = k_a + k_b, K = (k_b * k_...",D,3,,chemistry,stemez-PhysicalChemistry
501,4091,The following data show how the standard molar...,"[49.2 $\\mathrm{~kJ} \\mathrm{~mol}^{-1}$, 85....",E,4,,chemistry,scibench-matter
502,3747,The density of KF is 2.48 g/cm^3. The solid is...,"[4.2 × 10^23 formula units/cube, 3.5 × 10^23 f...",I,8,,chemistry,stemez-Chemistry
503,4323,A snowball at - 15°C is thrown at a large tree...,"[925 m/s, 1100 m/s, 720 m/s, 950 m/s, 892 m/s,...",F,5,,chemistry,stemez-Chemistry
504,3953,0.001 mol ofNaOHis added to 100 ml. of a solut...,"[5.20, 4.68, 5.10, 4.74, 4.76, 5.00, 4.70, 4.9...",E,4,,chemistry,stemez-Chemistry
...,...,...,...,...,...,...,...,...
995,10638,Based on the paper “SoK: SSL and HTTPS: Revisi...,[Valid DV certificates provide more confidence...,A,0,,computer science,ori_mmlu-computer_security
996,10444,A sorted list of numbers contains 500 elements...,"[500, 200, 50, 300, 250, 700, 100, 10, 400, 600]",H,7,,computer science,ori_mmlu-high_school_computer_science
997,10439,Explain what the computer takes as the attribu...,"[a) PERCENT(1:8)DEC, REAL, FLOAT(6); b) AMOUNT...",A,0,,computer science,stemez-ComputerScience
998,10709,Obtain the 9's and 10's complements of the fol...,"[s and ten, s and ten, s and ten, s and ten, s...",J,9,,computer science,stemez-ComputerScience


### Instantiate Ensemble class

Create an instance of the Ensemble class, this will be the workflow manager for the V-BReE framework.

Parameters:
- *client* - Object. API inference client
- *response_type* - String, *"logic"* or *"choice"*.  Define the type of response you want from the models.  *"logic"* will return logic only, *"choice"* will return both choice and logic.
- *verbose* - Boolean, default *False*. Control display of status messages.

In [22]:
e = v_bree.Ensemble(client = client, response_type = "choice", verbose = True)

### Check for available models

Use the HfApi module to review what models are available through the Hugging Face Inference Hub.

For the V-BReE project "text-generation" is the appropriate filter.  An author filter is recommended as well, otherwise the query will return many hundreds of responses.

Commonly chosen authors are:
- openai
- google
- meta-llama
- Qwen
- mistralai
- deepseek-ai

In [23]:
from huggingface_hub import HfApi

api = HfApi()
models = api.list_models(
    filter = "text-generation",
    author="meta-llama",
    limit=10
)
for model in models:
  print(model)

ModelInfo(id='meta-llama/Llama-3.1-8B-Instruct', author=None, sha=None, created_at=datetime.datetime(2024, 7, 18, 8, 56, tzinfo=datetime.timezone.utc), last_modified=None, private=False, disabled=None, downloads=6902650, downloads_all_time=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, likes=5510, library_name='transformers', tags=['transformers', 'safetensors', 'llama', 'text-generation', 'facebook', 'meta', 'pytorch', 'llama-3', 'conversational', 'en', 'de', 'fr', 'it', 'pt', 'hi', 'es', 'th', 'arxiv:2204.05149', 'base_model:meta-llama/Llama-3.1-8B', 'base_model:finetune:meta-llama/Llama-3.1-8B', 'license:llama3.1', 'eval-results', 'text-generation-inference', 'endpoints_compatible', 'region:us'], pipeline_tag='text-generation', mask_token=None, card_data=None, widget_data=None, model_index=None, config=None, transformers_info=None, trending_score=27, siblings=None, spaces=None, safetensors=None, security_repo_status=None, eval_results=None)
ModelInfo(i

### Add models to Ensemble

Add a string refrence for each model you want to inclue in the ensemble.  The format of these strings may vary depending on the client, and you may need to pre-load the models to your workspace if you are using them locally.

Pre-loading is not necessary for Hugging Face Inference Client.

There is no set limit to the number of models that can be added.  Prompting a single model configuration will return non-ensemble, single-prompt results.

In [24]:
e.add_model("openai/gpt-oss-20b:groq")
e.add_model("Qwen/Qwen2.5-7B-Instruct:together")
e.add_model("meta-llama/Llama-3.1-8B-Instruct:cerebras")
#NSCALE WAS OK
#NOT SAMBANOVA

### Run the V-BReE ensemble

To run the ensemble, pass:
- *data* - DataFrame. Properly formatted data frame.
- *id_col* - String. Column name of the unique identifier column.
- *question_col* - String. Column name of the question text column.
- *choices_col* - String. Column name of the list-formatted choices column.
- *domain_col* - String. Column name of the question domain column.
- *model_algorithm* - String, default *"order_added"*, *"order_added"* or *"random_start"*. Flag for whether the ensemble should always start with the first model added, or start with a random model.
- *temperature* - Float, default *0.0*. Desired temperature that the models should process at.

In [ ]:
e.run(data = sample,
      id_col = "question_id",
      question_col = "question",
      choices_col = "options",
      domain_col = "category",
      model_algorithm = "random_start",
      temperature = 0.0)

Processing question_id: 4205
Processing question_id: 4091
Error decoding JSON response: Invalid \uXXXX escape: line 1 column 2555 (char 2554)
Processing question_id: 3747
Error during API call: (Request ID: req_01kjqfa7rge4zbq61ben4dhex9)

Bad request:
{'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}
Prompt that caused the error: Question: The density of KF is 2.48 g/cm^3. The solid is made up of a cubic array of alternate K^+ and F^- ions at a spacing of 2.665 × 10^-8 cm. between centers. From these data, calculate the apparent value of the Avogadro number.
Existing Answer: To calculate the apparent value of the Avogadro number, we need to find the number of formula units in a unit cell of KF. Given that the density of KF is 2.48 g/cm^3 and the spacing between centers of alternate K^+ and F^- ions is 2.665 × 10^-8 cm, we can use the form

## Review the ensemble results

Retrieve the results from the ensemble.
- *selected_only* - Boolean. Flag indicating if you want the selected responses returned, or all responses processed by the ensemble.

In [ ]:
results = e.get_results(selected_only = False)

results.to_csv("v-bree_mmlupro_test_500_1000_20260302.csv")

display(results.head(5))

### Process the results as needed

Once the results are returned you can write to csv, check correctness of responses, generate plots, summary statistics, etc. - as with any other data.

In [ ]:
merged = pd.merge(results, sample[["question_id", "answer"]], left_on = "id", right_on = "question_id", how = "left")

import matplotlib.pyplot as plt

accuracy = (merged["answer"] == merged["selected_choice"]).mean()

plt.bar(x = ["Ensemble Accuracy"], height = [accuracy])
plt.figure(figsize=(10, 6))
plt.show()

## 4. Other Available V-BReE Methods

A variety of other methodsfor getting and setting various ensemble parameters are also available.

#### Setters:
*set_instructions(instructions: str)* - Set primary instructional prompt text.

*set_mcq_instructions(mcq_instructions: str)* - Set MCQ specific instructional prompt text.

*set_variance_threshold(threshold: float)* - Set the variance threshold starting value.

*set_variance_scaling_factor(scaling_factor: float)* - Set the variance threshold exponential scaling factor.

*set_variance_confidence_coefficient(coefficient: float)* - Set the variance confidence coefficient, for parameterizing the variance in the confidence formula.
    
*set_mean_confidence_coefficient(coefficient: float)* - Set the mean confidence coefficient, for parameterizing the mean in the confidence formula.
    
*set_n_confidence_coefficient(coefficient: float)* - Set the response count confidence coefficient, for parameterizing the response count in the confidence formula.


#### Getters:
*get_instructions()* - Get the current primary instructional prompt text.
    
*get_mcq_instructions()* - Get the current mcq specific instructional prompt text.
       
*get_variance_threshold()* - Get the current variance threshold starting value.

*get_variance_scaling_factor()* - Get the current variance scaling factor.
    
*get_confidence_coefficients()* - Get a dict of variance, mean and response count confidence coefficients.

In [ ]:
print(f"main instruction text: {e.get_instructions()}\n")
print(f"mcq instruction text: {e.get_mcq_instructions()}\n")
print(f"variance threshold: {e.get_variance_threshold()}\n")
print(f"variance scaling factor: {e.get_variance_scaling_factor()}\n")
print(f"confidence coefficients: {e.get_confidence_coefficients()}")